In [1]:
import google.generativeai as genai
import os

c:\Users\Administrator\Projects\neurosymbol_exp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\AppData\Local\Temp\ipykernel_17992\813451386.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

In [4]:
generation_config = {
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 40,
    "max_output_tokens": 200,
    "stop_sequences": ["\n\n"],
}

In [5]:
model = genai.GenerativeModel("gemma-4-31b-it", generation_config=generation_config)

In [10]:
response = model.generate_content("Write a haiku about the changing seasons.")
print(response.text)

*   Topic: Changing seasons.
    *   Format: Haiku (5-7-5 syllable structure).

    *   Autumn: Leaves, gold, red, falling, crisp air.
    *   Winter: Snow, ice, sleep, cold, white.
    *   Spring: Bloom, green, rain, awakening, buds.
    *   Summer: Sun, heat, long days, lush.
    *   Transition: Cycle, shift, turn, wheel.

    Green leaves turn to gold (5)
    Winter snow falls on the ground (7)
    Spring blooms once again (5)
    *Critique: A bit generic, but follows the rule.*

    Gold leaves drift away (5)
    White snow blankets all the earth (7)
    Green buds wake in spring (5)
    *Critique: Better imagery.*

    Green turns gold and red (5)
    Winter brings a silent sleep (7)
    Spring wakes up the world (5)
    *Critique: Stronger flow.*

    *Line 1:* Green turns gold and red (5)
    *Line 2:* Win-ter brings a si-lent sleep (7)
    *Line 3:* Spring wakes up the world (5)

    *Wait, let's try one that captures the "change" more poetically.*

    Green leaves turn to gol

In [6]:
import json, re
from pathlib import Path
from urllib.parse import urlparse, unquote

traffic_path = Path("../rule_based_tool/zap_scan_traffic.json")
with traffic_path.open("r", encoding="utf-8") as f:
    traffic = json.load(f)

def compact_request(record):
    req_hdr = record.get("requestHeader", "")
    req_body = record.get("requestBody", "")
    res_hdr = record.get("responseHeader", "")
    res_body = record.get("responseBody", "")

    first = req_hdr.splitlines()[0] if req_hdr else ""
    parts = first.split(" ")
    method = parts[0] if len(parts) > 0 else ""
    url = parts[1] if len(parts) > 1 else ""
    parsed = urlparse(url) if url else None

    status_line = res_hdr.splitlines()[0] if res_hdr else ""
    status_code = ""
    if status_line:
        status_parts = status_line.split(" ")
        if len(status_parts) >= 2 and status_parts[1].isdigit():
            status_code = int(status_parts[1])

    query = parsed.query if parsed else ""
    decoded_query = unquote(query)

    return {
        "id": record.get("id"),
        "method": method,
        "url": url,
        "path": parsed.path if parsed else "",
        "query": decoded_query,
        "body": req_body[:800] if req_body else "",
        "request_line": first,
        "response_status": status_code,
        "response_header": res_hdr[:800] if res_hdr else "",
        "response_body": res_body[:2000] if res_body else "",
    }

# Process all traffic without filtering
all_requests = [compact_request(rec) for rec in traffic]

print(f"Total requests loaded: {len(traffic)}")
print(f"Processed requests: {len(all_requests)}")
for row in all_requests[:5]:
    print({
        "id": row["id"],
        "method": row["method"],
        "url": row["url"],
        "status": row["response_status"],
    })

Total requests loaded: 178
Processed requests: 178
{'id': '3294', 'method': 'GET', 'url': 'http://juice-shop.herokuapp.com/7646048057944512000', 'status': 200}
{'id': '3295', 'method': 'GET', 'url': 'http://juice-shop.herokuapp.com/6580436607898199404', 'status': 200}
{'id': '3297', 'method': 'GET', 'url': 'http://juice-shop.herokuapp.com', 'status': 200}
{'id': '3298', 'method': 'GET', 'url': 'http://juice-shop.herokuapp.com/?id=%27&Submit=Submit', 'status': 200}
{'id': '3299', 'method': 'GET', 'url': 'http://juice-shop.herokuapp.com/?id=1%27&Submit=Submit', 'status': 200}


In [9]:
import time

def llm_sqli_label(sample, model):
    prompt = f"""You are a web security classifier focused on confirming successful SQL injection.
Decide whether this specific request appears to have SUCCESSFULLY exploited an SQL injection vulnerability.
Use both request and response evidence.

Return ONLY valid JSON with this exact schema:
{{
  \"injection_success\": true/false,
  \"vuln_confidence\": 0.0-1.0,
  \"technique\": \"error-based|union-based|boolean-based|time-based|unknown|none\",
  \"reason\": \"short explanation tied to evidence\",
  \"evidence\": [\"signal1\", \"signal2\"],
  \"response_signal\": \"error-message|content-diff|status-anomaly|none|unknown\",
  \"next_manual_check\": \"one practical verification step\"
}}

Important:
- injection_success=true only if response evidence suggests exploitation success.
- Payload intent alone is not enough.
- If uncertain, set injection_success=false and lower confidence.

Request and response data:
method={sample.get('method')}
url={sample.get('url')}
query={sample.get('query')}
request_body={sample.get('body')}
response_status={sample.get('response_status')}
response_header={sample.get('response_header')}
response_body_excerpt={sample.get('response_body')}
"""
    raw = model.generate_content(prompt).text.strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return {
            "injection_success": False,
            "vuln_confidence": 0.0,
            "technique": "unknown",
            "reason": "Model did not return JSON",
            "evidence": [],
            "response_signal": "unknown",
            "next_manual_check": "Replay request and compare baseline response.",
            "raw": raw[:400],
        }
    try:
        parsed = json.loads(raw[start:end+1])
        parsed["raw"] = raw[:400]
        return parsed
    except Exception:
        return {
            "injection_success": False,
            "vuln_confidence": 0.0,
            "technique": "unknown",
            "reason": "JSON parse failed",
            "evidence": [],
            "response_signal": "unknown",
            "next_manual_check": "Replay request and compare baseline response.",
            "raw": raw[:400],
        }

results = []
for i, sample in enumerate(all_requests):
    out = llm_sqli_label(sample, model)

    # Compatibility mapping for downstream HTML/report code.
    out["is_sqli"] = bool(out.get("injection_success", False))
    out["confidence"] = float(out.get("vuln_confidence", 0) or 0)

    out.update({
        "id": sample.get("id"),
        "method": sample.get("method"),
        "url": sample.get("url"),
        "response_status": sample.get("response_status"),
    })
    results.append(out)
    time.sleep(0.2)

confirmed = [r for r in results if r.get("injection_success") and float(r.get("vuln_confidence", 0) or 0) >= 0.6]
print(f"LLM analyzed: {len(results)}")
print(f"Likely successful injections: {len(confirmed)}")
for r in confirmed[:10]:
    print({
        "id": r["id"],
        "vuln_confidence": r.get("vuln_confidence"),
        "technique": r.get("technique"),
        "response_signal": r.get("response_signal"),
        "url": r["url"],
        "reason": r.get("reason"),
    })

with open("../rule_based_tool/sqli_llm_findings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("Saved -> ../rule_based_tool/sqli_llm_findings.json")

LLM analyzed: 178
Likely successful injections: 0
Saved -> ../rule_based_tool/sqli_llm_findings.json


In [10]:
from html import escape
from pathlib import Path
from datetime import datetime

# Prefer in-memory `results` from previous cell; fallback to saved JSON.
if "results" in globals() and isinstance(results, list):
    report_rows = results
else:
    with open("../rule_based_tool/sqli_llm_findings.json", "r", encoding="utf-8") as f:
        report_rows = json.load(f)

total = len(report_rows)
flagged_rows = [r for r in report_rows if r.get("is_sqli") and float(r.get("confidence", 0) or 0) >= 0.6]
high_rows = [r for r in report_rows if r.get("is_sqli") and float(r.get("confidence", 0) or 0) >= 0.85]
medium_rows = [r for r in report_rows if r.get("is_sqli") and 0.6 <= float(r.get("confidence", 0) or 0) < 0.85]
flagged = len(flagged_rows)
rate = (flagged / total * 100) if total else 0.0

def classify_row(r):
    conf = float(r.get("confidence", 0) or 0)
    if r.get("is_sqli") and conf >= 0.85:
        return "high", "High"
    if r.get("is_sqli") and conf >= 0.6:
        return "med", "Medium"
    return "low", "Low"

rows_html = []
for r in sorted(report_rows, key=lambda x: float(x.get("confidence", 0) or 0), reverse=True):
    cls, sev_label = classify_row(r)
    conf = float(r.get("confidence", 0) or 0)
    evidence = ", ".join(map(str, r.get("evidence", [])))
    verdict = "Likely SQLi" if r.get("is_sqli") else "Benign/Unclear"
    rows_html.append(
        f"<tr class='{cls}'>"
        f"<td><span class='id'>{escape(str(r.get('id', '')))}</span></td>"
        f"<td><span class='method'>{escape(str(r.get('method', '')))}</span></td>"
        f"<td class='url'>{escape(str(r.get('url', '')))}</td>"
        f"<td><span class='pill sev-{cls}'>{sev_label}</span></td>"
        f"<td><span class='pill verdict'>{verdict}</span></td>"
        f"<td><span class='pill conf'>{conf:.2f}</span></td>"
        f"<td>{escape(str(r.get('technique', '')))}</td>"
        f"<td>{escape(str(r.get('reason', '')))}</td>"
        f"<td>{escape(evidence)}</td>"
        "</tr>"
    )

generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
html_report = f"""<!DOCTYPE html>
<html lang='en'>
<head>
  <meta charset='utf-8' />
  <meta name='viewport' content='width=device-width, initial-scale=1' />
  <title>SQLi LLM Findings</title>
  <style>
    :root {{
      --bg: #f4f7fb;
      --paper: #ffffff;
      --ink: #1b2a41;
      --muted: #607087;
      --line: #d9e3ee;
      --accent: #0f766e;
      --accent-2: #164e63;
      --high-bg: #ffe8e8;
      --high-ink: #8a1f1f;
      --med-bg: #fff4dd;
      --med-ink: #8a5a00;
      --low-bg: #e9f8ef;
      --low-ink: #1f6b3a;
    }}

    * {{ box-sizing: border-box; }}
    body {{
      margin: 0;
      color: var(--ink);
      background: radial-gradient(1400px 700px at 20% -10%, #d8f0ec 0%, transparent 60%),
                  radial-gradient(1400px 700px at 100% 0%, #dbe8ff 0%, transparent 55%),
                  var(--bg);
      font-family: "Segoe UI", "Trebuchet MS", Tahoma, sans-serif;
    }}

    .wrap {{
      max-width: 1320px;
      margin: 28px auto 40px;
      padding: 0 16px;
    }}

    .hero {{
      background: linear-gradient(130deg, var(--accent-2), var(--accent));
      color: #fff;
      border-radius: 16px;
      padding: 22px 24px;
      box-shadow: 0 12px 28px rgba(22, 78, 99, 0.28);
      margin-bottom: 14px;
    }}
    .hero h1 {{ margin: 0 0 8px; font-size: 28px; letter-spacing: 0.3px; }}
    .hero .sub {{ margin: 0; font-size: 14px; opacity: 0.95; }}

    .grid {{
      display: grid;
      grid-template-columns: repeat(5, minmax(140px, 1fr));
      gap: 10px;
      margin-bottom: 14px;
    }}
    .stat {{
      background: var(--paper);
      border: 1px solid var(--line);
      border-radius: 12px;
      padding: 12px 14px;
      box-shadow: 0 4px 14px rgba(27, 42, 65, 0.06);
    }}
    .stat .k {{ color: var(--muted); font-size: 12px; }}
    .stat .v {{ margin-top: 2px; font-size: 24px; font-weight: 700; }}

    .panel {{
      background: var(--paper);
      border: 1px solid var(--line);
      border-radius: 14px;
      box-shadow: 0 6px 18px rgba(27, 42, 65, 0.07);
      overflow: hidden;
    }}
    .legend {{
      display: flex;
      flex-wrap: wrap;
      gap: 10px;
      padding: 12px 14px;
      border-bottom: 1px solid var(--line);
      background: #f8fbff;
      font-size: 13px;
      color: var(--muted);
    }}
    .legend .item {{ display: inline-flex; align-items: center; gap: 6px; }}
    .dot {{ width: 11px; height: 11px; border-radius: 50%; display: inline-block; }}
    .dot.high {{ background: #ef4444; }}
    .dot.med {{ background: #d97706; }}
    .dot.low {{ background: #22c55e; }}

    .table-wrap {{ overflow-x: auto; }}
    table {{ width: 100%; border-collapse: separate; border-spacing: 0; font-size: 13px; }}
    th, td {{
      padding: 10px 9px;
      border-bottom: 1px solid var(--line);
      text-align: left;
      vertical-align: top;
    }}
    th {{
      position: sticky;
      top: 0;
      z-index: 1;
      background: #eef4fb;
      color: #28405f;
      font-size: 12px;
      text-transform: uppercase;
      letter-spacing: 0.4px;
    }}
    tbody tr:hover {{ filter: brightness(0.985); }}
    tr.high {{ background: var(--high-bg); }}
    tr.med {{ background: var(--med-bg); }}
    tr.low {{ background: var(--low-bg); }}

    .pill {{
      display: inline-block;
      border-radius: 999px;
      padding: 3px 9px;
      font-weight: 600;
      font-size: 11px;
      white-space: nowrap;
    }}
    .sev-high {{ background: #ffd4d4; color: var(--high-ink); }}
    .sev-med {{ background: #ffe8bf; color: var(--med-ink); }}
    .sev-low {{ background: #d7f5e3; color: var(--low-ink); }}
    .verdict {{ background: #e7eef7; color: #27415f; }}
    .conf {{ background: #e8f7f2; color: #145f56; }}

    .id, .method {{ font-weight: 600; }}
    .url {{ min-width: 360px; max-width: 560px; word-break: break-all; }}

    @media (max-width: 1060px) {{
      .grid {{ grid-template-columns: repeat(2, minmax(140px, 1fr)); }}
      .hero h1 {{ font-size: 24px; }}
    }}
    @media (max-width: 640px) {{
      .grid {{ grid-template-columns: 1fr; }}
    }}
  </style>
</head>
<body>
  <div class='wrap'>
    <section class='hero'>
      <h1>SQL Injection Detection Report</h1>
      <p class='sub'>Generated at {generated_at} from LLM classification output.</p>
    </section>

    <section class='grid'>
      <div class='stat'><div class='k'>Total analyzed</div><div class='v'>{total}</div></div>
      <div class='stat'><div class='k'>Flagged (>= 0.60)</div><div class='v'>{flagged}</div></div>
      <div class='stat'><div class='k'>Flag rate</div><div class='v'>{rate:.1f}%</div></div>
      <div class='stat'><div class='k'>High confidence</div><div class='v'>{len(high_rows)}</div></div>
      <div class='stat'><div class='k'>Medium confidence</div><div class='v'>{len(medium_rows)}</div></div>
    </section>

    <section class='panel'>
      <div class='legend'>
        <span class='item'><span class='dot high'></span>High confidence SQLi</span>
        <span class='item'><span class='dot med'></span>Medium confidence SQLi</span>
        <span class='item'><span class='dot low'></span>Low / benign</span>
      </div>
      <div class='table-wrap'>
        <table>
          <thead>
            <tr>
              <th>ID</th>
              <th>Method</th>
              <th>URL</th>
              <th>Severity</th>
              <th>Verdict</th>
              <th>Confidence</th>
              <th>Technique</th>
              <th>Reason</th>
              <th>Evidence</th>
            </tr>
          </thead>
          <tbody>
            {''.join(rows_html)}
          </tbody>
        </table>
      </div>
    </section>
  </div>
</body>
</html>"""

out_html = Path("../rule_based_tool/sqli_llm_findings.html")
out_html.write_text(html_report, encoding="utf-8")
print(f"Saved HTML report -> {out_html}")

Saved HTML report -> ..\rule_based_tool\sqli_llm_findings.html
